# kaggle run
- https://code.visualstudio.com/docs/datascience/jupyter-notebooks#_connect-to-a-remote-jupyter-server
- 

In [2]:
!pwd

/kaggle/working


In [3]:
!pip install torch tokenizers sympy
!git clone https://github.com/yongsingyou/reasoning-from-scratch.git
import sys

sys.path.append("/kaggle/working/reasoning-from-scratch")

fatal: destination path 'reasoning-from-scratch' already exists and is not an empty directory.


In [4]:
from reasoning_from_scratch.qwen3 import download_qwen3_small

# Downloads model weights (~1.5 GB)
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [5]:
# understand tokenizer
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

In [6]:
# load the model
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B
# Qwen3Model is the pytorch implementation of the model. and QWEN_CONFIG_06_B is the config.
# TODO: learn it.

device = get_device()
model_path = Path("qwen3") / "qwen3-0.6B-base.pth"
model = Qwen3Model(QWEN_CONFIG_06_B)  # model still random weight.
model.load_state_dict(torch.load(model_path))
model.to(device)

Using NVIDIA CUDA GPU


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [7]:
# evaluate baseline model
from reasoning_from_scratch.ch03 import (
    evaluate_math500_stream,
    load_math500_test,
)

math500 = load_math500_test()

num_correct, num_examples, acc = evaluate_math500_stream(
    model=model,
    tokenizer=tokenizer,
    device=device,
    math_data=math500[
        :50
    ],  # use [:50] for a quick eval, remove slice for full 500
    max_new_tokens=512,
    verbose=True,
)

print(f"Accuracy: {acc:.3f} ({num_correct}/{num_examples})")


 \boxed{(3,\frac{\pi}{2})}MATH-500: 1/50 | ETA: 54s         

MATH-500: 1/50 | ETA: 54s         
Extracted: (3,\frac{\pi}{2})
Expected:  \left( 3, \frac{\pi}{2} \right)
Correct so far: 1
--------------------------------------------------
 \[\boxed{p^2 - 3pq + 2q^2}\]MATH-500: 2/50 | ETA: 54s         

MATH-500: 2/50 | ETA: 54s         
Extracted: p^2 - 3pq + 2q^2
Expected:  p - q
Correct so far: 1
--------------------------------------------------
 \boxed{-\frac{1}{2}}MATH-500: 3/50 | ETA: 42s         

MATH-500: 3/50 | ETA: 42s         
Extracted: -\frac{1}{2}
Expected:  \frac{14}{3}
Correct so far: 1
--------------------------------------------------
 196 has 16 positive whole-number divisors.MATH-500: 4/50 | ETA: 38s         

MATH-500: 4/50 | ETA: 38s         
Extracted: 16
Expected:  9
Correct so far: 1
--------------------------------------------------
 To determine which student has the greatest average speed, we need to calculate the average speed for each student by dividing t

In [8]:
# 6.11 training loop
import time
import torch
import csv
from pathlib import Path

from reasoning_from_scratch.ch03 import (
    load_math500_test,
)

from reasoning_from_scratch.ch06 import compute_grpo_loss

math_train = load_math500_test()
math_train[0]


CSV_PATH = Path("/kaggle/working/grpo_metrics.csv")


def grpo_train(steps, lr=1e-5, num_rollouts=4, max_new_tokens=256):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0

    # Write CSV header
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    with CSV_PATH.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(
            [
                "step",
                "total_steps",
                "loss",
                "reward_avg",
                "tokens_per_sec",
                "avg_response_len",
            ]
        )

    try:
        for step in range(steps):
            step_start = time.perf_counter()
            current_step = step + 1
            example = math_train[step % len(math_train)]

            stats = compute_grpo_loss(
                model,
                tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=0.8,
                top_p=0.9,
            )

            optimizer.zero_grad()
            stats["loss_tensor"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_time = time.perf_counter() - step_start
            step_tokens = sum(s["gen_len"] for s in stats["samples"])
            avg_response_len = (
                step_tokens / len(stats["samples"]) if stats["samples"] else 0.0
            )
            tok_per_sec = step_tokens / step_time if step_time > 0 else 0.0

            # Append to CSV
            with CSV_PATH.open("a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(
                    [
                        current_step,
                        steps,
                        stats["loss"],
                        reward_avg,
                        tok_per_sec,
                        avg_response_len,
                    ]
                )

            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"tok/sec={tok_per_sec:.1f}"
            )

    except KeyboardInterrupt:
        print(f"Stopped at step {current_step}.")

    return model


# Run training -- adjust steps/max_new_tokens as needed
torch.manual_seed(42)
trained_model = grpo_train(
    steps=10, lr=1e-5, num_rollouts=4, max_new_tokens=256
)


[Step 1/10] loss=-4.6847 reward_avg=0.500 tok/sec=18.5
[Step 2/10] loss=-0.0000 reward_avg=0.000 tok/sec=19.1
[Step 3/10] loss=0.3418 reward_avg=0.250 tok/sec=13.3
[Step 4/10] loss=-0.0000 reward_avg=0.000 tok/sec=18.8
Stopped at step 5.


In [ ]:
# evaluate trained model
from reasoning_from_scratch.ch03 import (
    evaluate_math500_stream,
    load_math500_test,
)

math500 = load_math500_test()

num_correct, num_examples, acc = evaluate_math500_stream(
    model=trained_model,
    tokenizer=tokenizer,
    device=device,
    math_data=math500[
        :50
    ],  # use [:50] for a quick eval, remove slice for full 500
    max_new_tokens=512,
    verbose=True,
)

print(f"Accuracy: {acc:.3f} ({num_correct}/{num_examples})")

In [ ]:
# save checkpoint
torch.save(trained_model.state_dict(), "/kaggle/working/qwen3-grpo-trained.pth")
print("Checkpoint saved.")

In [ ]:
# visualize the train process
from reasoning_from_scratch.ch07 import plot_grpo_metrics

plot_grpo_metrics(
    CSV_PATH, columns=["loss", "reward_avg", "avg_response_len", "eval_acc"]
)

# chapter 7

In [9]:
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import load_model_and_tokenizer

device = get_device()
model, tokenizer_base = load_model_and_tokenizer(
    which_model="base", device=device, use_compile=False
)


Using NVIDIA CUDA GPU
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [10]:
print(tokenizer_base.encode("<think>"))

[13708, 766, 29]


In [11]:
tokenizer_base._tok.add_special_tokens(
    ["<tool_response>", "</tool_response>", "<think>", "</think>"]
)

4

In [13]:
tokenizer_base.decode([151668])

'</think>'

In [ ]:
# modify reward model to encourage model to behave certain way. like outputting <think> token.
#
def reward_format(
    token_ids, prompt_len, start_think_id=151667, end_think_id=151668
):
    try:
        gen = token_ids[prompt_len:]
        return float(gen.index(start_think_id) < gen.index(end_think_id))
    except ValueError:
        return 0.0


prompt = "Calculate ..."
rollout = "Let's ... <think> ... </think> ..."
token_ids = tokenizer_base.encode(prompt + rollout)
prompt_len = len(tokenizer_base.encode(prompt))
reward_format(token_ids, prompt_len, start_think_id=151667, end_think_id=151668)

1.0

In [39]:
!pip install -e /kaggle/working/reasoning-from-scratch

!uv run reasoning-from-scratch/ch07/03_rlvr_grpo_scripts_advanced/7_6_plus_format_reward.py --steps 4 --max_new_tokens 256

4952.32s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
Obtaining file:///kaggle/working/reasoning-from-scratch
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 88.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 70.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [34]:
!ls reasoning-from-scratch

4701.21s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
ch01  ch05  chC  chG		 reasoning_from_scratch
ch02  ch06  chD  LICENSE	 requirements.txt
ch03  ch07  chE  pyproject.toml  tests
ch04  ch08  chF  README.md	 troubleshooting.md


In [37]:
sys.path.append("/kaggle/working/reasoning-from-scratch/reasoning_from_scratch")